# Module 07 — Mask R-CNN & Instance Segmentation

Instance segmentation assigns a unique mask to each individual object instance.

In [ ]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from mask_head import MaskHead, paste_masks_in_image
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Mask R-CNN Architecture

Mask R-CNN extends Faster R-CNN with a parallel mask prediction head.

In [ ]:
# Inspect the mask head architecture
mask_head = MaskHead(in_channels=256, roi_size=14, num_classes=80)
print(mask_head)

# Test forward pass
roi_feats = torch.randn(4, 256, 14, 14)
mask_logits = mask_head(roi_feats)
print('Mask logits shape:', mask_logits.shape)  # (4, 80, 28, 28)

## 2. Inference with Pretrained Mask R-CNN

In [ ]:
model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
model.eval().to(DEVICE)

import urllib.request, io
COCO_CLASSES = ['__bg__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane',
                'bus', 'train', 'truck', 'boat', 'traffic light']

URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/Kittyply_edit1.jpg/320px-Kittyply_edit1.jpg'
try:
    with urllib.request.urlopen(URL) as r: data = r.read()
    img = Image.open(io.BytesIO(data)).convert('RGB')
except:
    img = Image.fromarray(np.random.randint(50,200,(400,500,3),dtype=np.uint8))

img_t = TF.to_tensor(img).to(DEVICE)
with torch.no_grad():
    preds = model([img_t])[0]

print(f'Detected {len(preds["boxes"])} objects')
print(f'Mask shape: {preds["masks"].shape}')

## 3. Visualising Predictions

In [ ]:
img_np = np.array(img)
overlaid = img_np.copy()

threshold = 0.5
colors = plt.cm.rainbow(np.linspace(0,1,len(preds['boxes'])))

for i, (box, score, mask, label) in enumerate(
    zip(preds['boxes'], preds['scores'], preds['masks'], preds['labels'])
):
    if score < threshold: continue
    m = mask[0].cpu().numpy() > 0.5
    color = (np.array(colors[i][:3]) * 255).astype(np.uint8)
    overlaid[m] = (0.5 * overlaid[m] + 0.5 * color).astype(np.uint8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(img_np); ax1.set_title('Original'); ax1.axis('off')
ax2.imshow(overlaid); ax2.set_title('Mask R-CNN predictions'); ax2.axis('off')
plt.tight_layout(); plt.show()

## 4. Fine-tuning on Custom Data

For a 2-class custom dataset (background + 1 class):

In [ ]:
def get_instance_segmentation_model(num_classes):
    """Load Mask R-CNN and replace heads for num_classes classes."""
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    # Box head
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    # Mask head
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model

num_classes = 2  # background + 1
custom_model = get_instance_segmentation_model(num_classes)
print('Model adapted for', num_classes, 'classes')
print('Trainable parameters:', sum(p.numel() for p in custom_model.parameters() if p.requires_grad)/1e6, 'M')

## Exercise — Mask IoU

Implement `mask_iou(pred_mask, gt_mask)` for two binary mask tensors.

In [ ]:
### EXERCISE
def mask_iou(pred_mask: torch.Tensor, gt_mask: torch.Tensor) -> float:
    """
    Compute IoU between two binary masks.
    pred_mask, gt_mask: bool tensors of any shape.
    Returns float in [0, 1].
    """
    # TODO
    raise NotImplementedError